# 2: Exploratory Data Analysis

Loads the fully merged OTP + GTFS dataset and runs exploratory analysis.  
Run after `0_otp_load_and_clean.py` and `1_gtfs_merge.py`.

In [ ]:
# Library installs:
# pandas        : standard data analysis package
# numpy         : scientific + math computation
# matplotlib    : base plotting library
# seaborn       : additional plotting capabilities

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

BASEPATH = "data"  # local path to data directory (update as needed)

## Load Data

In [ ]:
# Load from BASEPATH -- output of 1_gtfs_merge.py
df_valid = pd.read_parquet(f"{BASEPATH}/df_gtfs_linked.parquet")
print(f"Loaded {len(df_valid):,} rows")

## Quick Exploration

In [ ]:
df_valid.head()

In [ ]:
df_valid.dtypes

In [ ]:
# date range covered
print(f"Earliest date: {df_valid['service_date'].min().date()}")
print(f"Latest date:   {df_valid['service_date'].max().date()}")
print(f"Span: {(df_valid['service_date'].max() - df_valid['service_date'].min()).days // 365} years")

In [ ]:
# basic stats on lateness
print(df_valid["lateness"].describe().apply(lambda x: f"{x:.2f}"))

## Lateness Distribution

In [ ]:
bins = [-999, 0, 2, 6, 10, 20, np.inf]
labels = ["On time", "0–2 min", "2–6 min",
          "6–10 min", "10–20 min", "20+ min"]

df_valid["lateness_bucket"] = pd.cut(df_valid["lateness"], bins = bins, labels = labels)

bucket_counts = df_valid["lateness_bucket"].value_counts().reindex(labels)

fig, ax = plt.subplots()
ax.bar(labels, bucket_counts,
       color = np.array(["seagreen"]*3 + ["firebrick"]*3),
       edgecolor = "white")
ax.set_title("Train delays (all years)", fontsize = 14)
ax.set_ylabel("Number of observations")
ax.set_xlabel("")
ax.set_xticks(range(len(labels)))
ax.tick_params(axis = "x", rotation = 0)
plt.tight_layout()
plt.show()

In [ ]:
# Plotting distribution of delays
fig, (ax1, ax2) = plt.subplots(2, 1, figsize = (10, 8))

# Base scale, capped at 60m, where we capture most of the values
df_valid[df_valid["lateness"] <= 60]["lateness"].hist(
    bins = 60, ax = ax1, color = "steelblue", edgecolor = "white")
ax1.set_title("Lateness distribution (0–60 min)")
ax1.set_xlabel("Minutes late")
ax1.set_ylabel("Observations")

# Log scale, where we can see some of the extreme values
df_valid["lateness"].hist(
    bins = 100, ax = ax2, color = "steelblue", edgecolor = "white")
ax2.set_yscale("log")
ax2.set_title("Full distribution (log scale)")
ax2.set_xlabel("Minutes late")
ax2.set_ylabel("Observations (log scale)")

plt.tight_layout()
plt.show()

Sharp dropoff after 25 minutes in the top plot shows really long delays are pretty rare. Extreme delays in the bottom plot are likely a handful of specific incidents and not any consistent pattern (so I consider them valid).  
Modeling should likely be done on log-transformed data given the extreme right tail or binary on-time / late target.

## OTP Over Time

In [ ]:
# Monthly OTP rate: % of observations where train was within 6 min
df_valid["year_month"] = df_valid["service_date"].dt.to_period("M")
df_valid["on_time"] = df_valid["lateness"] < 6  # SEPTA threshold

monthly_otp = df_valid.groupby("year_month")["on_time"].mean() * 100

fig, ax = plt.subplots(figsize = (14, 5))
monthly_otp.plot(ax = ax, linewidth = 1, color = "steelblue")
ax.axhline(91, color = "red", linestyle = "--", linewidth = 1, label = "SEPTA 91% target")
ax.set_title("Monthly on-time performance (% of observations)", fontsize = 14)
ax.set_ylabel("% on time")
ax.set_xlabel("")
ax.legend()
plt.tight_layout()
plt.show()

## Time of Day Analysis

In [ ]:
# Note: service_date is not the same as actual calendar date.
# SEPTA service day runs 3 AM to 3 AM next day, so trains running
# between midnight and 2:59 AM are on the prior service date.
# We treat service_date as "date" for all analyses to keep w/ SEPTA standards,
# but this is something to bear in mind when interpreting results.

time_bins = ["00:00:00", "03:00:00", "06:00:00", "09:00:00",
             "16:00:00", "19:00:00", "23:59:59"]
time_labels = ["Late Night (12-2)", "Early Morning (3-5)", "AM Peak (6-8)",
               "Mid-day (9-3)", "PM Peak (4-6)", "Evening (7-11)"]

df_valid["time_of_day"] = pd.cut(df_valid["time"],
                                  bins = time_bins,
                                  labels = time_labels)

df_latenight = df_valid[df_valid["time"] < "03:00:00"]
print(f"Service date != Actual date observations: {len(df_latenight):,}")
df_latenight.head()

In [ ]:
# OTP by time of day over time
monthly_tod = (
    df_valid
    .groupby(["year_month", "time_of_day"], 
             observed = True)["lateness"]
    .mean()
    .unstack("time_of_day"))

fig, ax = plt.subplots(figsize = (12, 4))
monthly_tod.plot(ax = ax, linewidth = 1.5)
ax.set_title("Average monthly delay by time of day", fontsize = 14)
ax.set_ylabel("Avg minutes late")
ax.set_xlabel("")
ax.axhline(6, color = "red", linestyle = "--", 
           linewidth = 1, label = "SEPTA OTP cutoff (6 min)")
ax.legend()
plt.tight_layout()
plt.show()

# Summary stats by time of day
for label in time_labels:
    subset = df_valid[df_valid["time_of_day"] == label]
    print(f"\n{label}")
    print(f"  Total observations: {len(subset):,}")
    print(f"  Avg delay:          {subset['lateness'].mean():.1f} min")
    print(f"  On-time rate:       {(subset['lateness'] < 6).mean()*100:.1f}%")

## Line-level Analysis

In [ ]:
# Remove "Line" suffix from line names for cleaner labels
df_valid["line"] = df_valid["line"].str.replace(" Line$", "", regex = True)

In [ ]:
# Average lateness and OTP rate by line
# Filtered to normal coverage periods only
# Sorted worst -> best OTP rate
line_stats = (
    df_valid[df_valid["period_flag"] == "normal"]
    .groupby("line")["lateness"]
    .agg(
        otp_rate = lambda x: (x < 6).mean() * 100,
        avg_lateness = "mean",
        observations = "count"
    )
    .sort_values("otp_rate", ascending = True)
    .round(2)
)
display(line_stats)

- At first glance, the longest lines have the worst OTP and vice versa. Will need to control for distance in modeling.
- Amtrak track-sharing may explain underperformance on Paoli/Thorndale, Wilmington/Newark, and West Trenton, though distance could be a strong confound.
- Lateness and OTP track closely across lines.

In [ ]:
# OTP rate by line over time
line_otp = (
    df_valid[df_valid["period_flag"] == "normal"]
    .groupby(["year", "line"])
    .agg(
        otp_rate = ("lateness", lambda x: (x < 6).mean() * 100),
        obs = ("lateness", "count")
    )
    .reset_index()
)

# Filter out line-years with very low observations (e.g. Media/Elwyn 2024)
line_otp = line_otp[line_otp["obs"] >= 1000]

fig, ax = plt.subplots(figsize = (14, 5))
for line, group in line_otp.groupby("line"):
    ax.plot(group["year"], group["otp_rate"], marker = "o", linewidth = 1.5, label = line)
ax.axhline(91, color = "red", linestyle = "--", linewidth = 1, label = "SEPTA 91% target")
ax.set_title("OTP rate by line over time")
ax.set_ylabel("% on time")
ax.set_xlabel("")
ax.legend(bbox_to_anchor = (1.01, 1), loc = "upper left", fontsize = 8)
plt.tight_layout()
plt.show()

Observations:
- **Cynwyd** is SEPTA's shortest line, and the best-performing. It's also the only one to reach SEPTA's 91% target (in 2020).
- **Paoli/Thorndale** is consistently one of the lowest performers, possibly supporting the Amtrak track-sharing hypothesis.
- **2020** shows a bump across all lines, likely due to decreased ridership/service during COVID.
- **2024–2025** shows a sharp collapse in OTP across the board, likely reflecting the SEPTA funding crisis - service cuts, fare increases, and infrastructure stress after pandemic relief funding ended. Emergency 20% service cuts in Aug 2025 (partially restored after Shapiro approved emergency funding).

In [ ]:
# Average delay by line and time of day
line_time_of_day = (
    df_valid[df_valid["period_flag"] == "normal"]
    .groupby(["line", "time_of_day"], observed = True)["lateness"]
    .mean()
    .unstack("time_of_day")
    .round(2)
)
display(line_time_of_day)